In [3]:
API_KEY = "bd0ed2582afeb72e616e24d0ec65bb360f416ebd"

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display

# 1. Configuración de ZenRows
ZENROWS_API_KEY = 'bd0ed2582afeb72e616e24d0ec65bb360f416ebd'
URL_MERCADO_LIBRE = 'https://listado.mercadolibre.com.mx/laptops'

api_url = "https://api.zenrows.com/v1/"
params = {
    "url": URL_MERCADO_LIBRE,
    "apikey": ZENROWS_API_KEY,
    "js_render": "true",      # Ejecuta JavaScript de forma segura
    "antibot": "true",        # Resuelve Cloudflare/Datadome
    "premium_proxy": "true"   # Usa IPs residenciales premium
}

print("Enviando petición a través de ZenRows (Anti-bot activado)...")
response = requests.get(api_url, params=params, timeout=60)

print(f"Status de ZenRows: {response.status_code}")

if response.status_code == 200:
    print("¡Conexión exitosa! Analizando el HTML...\n")
    soup = BeautifulSoup(response.text, 'lxml')
    
    lista_laptops = []
    
    # Extraemos usando las clases correctas
    titulos = soup.find_all('a', class_='poly-component__title')
    precios = soup.find_all('div', class_='poly-component__price')
    
    for titulo, precio in zip(titulos, precios):
        lista_laptops.append({
            'Modelo / Especificaciones': titulo.text.strip(),
            'Precio': precio.text.strip()
        })
    
    if lista_laptops:
        # 2. Creación y exportación con Pandas
        df = pd.DataFrame(lista_laptops)
        df.to_csv('laptops_mercadolibre.csv', index=False, encoding='utf-8')
        
        print(f"¡Victoria! Se extrajeron y guardaron {len(df)} laptops en tu archivo CSV.")
        display(df.head())
    else:
        print("La página cargó, pero no se encontraron productos. Podría ser un cambio en el HTML.")
        
        # Guardamos el HTML para revisarlo si algo falla
        with open("error_zenrows.html", "w", encoding="utf-8") as f:
            f.write(response.text)
        print("Se guardó 'error_zenrows.html' para diagnóstico.")
else:
    print(f"Hubo un error con la API: {response.text}")

Enviando petición a través de ZenRows (Anti-bot activado)...
Status de ZenRows: 200
¡Conexión exitosa! Analizando el HTML...

La página cargó, pero no se encontraron productos. Podría ser un cambio en el HTML.
Se guardó 'error_zenrows.html' para diagnóstico.


In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display

url = 'https://listado.mercadolibre.com.mx/laptops'

# ¡El disfraz definitivo! Nos hacemos pasar por los servidores de indexación de Google
headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

print("Consultando Mercado Libre como si fuéramos Google...")
response = requests.get(url, headers=headers, timeout=20)

print(f"Status: {response.status_code}")

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'lxml')
    
    # Comprobamos si nos mandaron al Login Wall
    if "ingresa a tu cuenta" in response.text.lower():
         print("Mercado Libre detectó el truco y nos mandó al login wall.")
    else:
        lista_laptops = []
        titulos = soup.find_all('a', class_='poly-component__title')
        precios = soup.find_all('div', class_='poly-component__price')
        
        for titulo, precio in zip(titulos, precios):
            # Buscamos específicamente el número del precio
            span_precio = precio.find('span', class_='andes-money-amount__fraction')
            
            # Si lo encuentra, sacamos el texto. Si no, dejamos un mensaje de error o el texto completo
            precio_limpio = span_precio.text.strip() if span_precio else "No disponible"
            
            lista_laptops.append({
                'Modelo': titulo.text.strip(),
                'Precio ($)': precio_limpio
            })
            
        if lista_laptops:
            df = pd.DataFrame(lista_laptops)
            df.to_csv('laptops_mercadolibre_googlebot.csv', index=False)
            print(f"¡Éxito! Extraímos {len(df)} laptops saltándonos la seguridad.")
            display(df.head())
        else:
             print("Página cargada, pero sin productos. (Posible cambio de clases HTML).")

Consultando Mercado Libre como si fuéramos Google...
Status: 200
¡Éxito! Extraímos 48 laptops saltándonos la seguridad.


,Modelo,Precio ($)
0,Laptop HP 250 G10 Intel Core i3 N305 16GB 256G...,"13,999"
1,Mac Mini M4 24gb (memoria Unificada) 512 gb Ap...,"28,999"
2,Ultrabook Intel N150 Machcreator 15a 15.6 Pulg...,"8,138"
3,Laptop HP 245 AMD Ryzen 3 7330U 16GB 512GB SSD...,"11,999"
4,Laptop Asus Vivobook Intel Core I3 8gb Ram 128...,"7,999"


In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
import time

# 1. Configuración del "Disfraz" de Googlebot
headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

url_actual = 'https://listado.mercadolibre.com.mx/laptops'
lista_laptops = []
paginas_a_extraer = 3  # Vamos a extraer las primeras 3 páginas (puedes aumentar esto luego)

print(f"Iniciando extracción automática de {paginas_a_extraer} páginas...\n")

# 2. Bucle de Paginación
for pagina in range(paginas_a_extraer):
    print(f"-> Escaneando página {pagina + 1}...")
    response = requests.get(url_actual, headers=headers, timeout=20)
    
    if response.status_code != 200 or "ingresa a tu cuenta" in response.text.lower():
        print("¡Alerta! Mercado Libre bloqueó la conexión en esta página.")
        break
        
    soup = BeautifulSoup(response.text, 'lxml')
    
    titulos = soup.find_all('a', class_='poly-component__title')
    precios = soup.find_all('div', class_='poly-component__price')
    
    # 3. Extracción de detalles por producto
    for titulo, precio in zip(titulos, precios):
        
        # Extraemos el texto del precio limpio (ej. "9,899")
        span_precio = precio.find('span', class_='andes-money-amount__fraction')
        precio_texto = span_precio.text.strip() if span_precio else "0"
        
        # Extraemos el enlace del producto (atributo 'href')
        enlace = titulo.get('href', 'Sin enlace')
        
        lista_laptops.append({
            'Modelo': titulo.text.strip(),
            'Precio ($)': precio_texto,
            'Enlace': enlace
        })
        
    # 4. Buscar el botón "Siguiente" para la próxima iteración
    boton_siguiente = soup.find('a', title='Siguiente')
    if boton_siguiente and boton_siguiente.get('href'):
        url_actual = boton_siguiente.get('href') # Actualizamos la URL para la próxima vuelta
        time.sleep(2) # ¡IMPORTANTE! Pausa de 2 segundos para simular ser un humano leyendo
    else:
        print("No se encontró el botón 'Siguiente' o llegamos al final.")
        break

# 5. Procesamiento y Limpieza con Pandas
if lista_laptops:
    df = pd.DataFrame(lista_laptops)
    
    # Data Cleaning: Quitamos las comas de los precios y los convertimos a números enteros
    # '9,899' -> '9899' -> 9899
    df['Precio ($)'] = df['Precio ($)'].str.replace(',', '', regex=False).astype(int)
    
    # Guardamos el CSV definitivo
    df.to_csv('laptops_mercadolibre_dataset.csv', index=False, encoding='utf-8')
    
    print(f"\n¡Éxito! Se han extraído {len(df)} laptops en total y se guardaron en 'laptops_mercadolibre_dataset.csv'")
    
    # Como ahora son números, podemos pedirle a Pandas que nos muestre las 5 más baratas
    print("\nLas 5 laptops más baratas encontradas son:")
    display(df.sort_values(by='Precio ($)', ascending=True).head())
else:
    print("No se extrajeron datos.")

Iniciando extracción automática de 3 páginas...

-> Escaneando página 1...
No se encontró el botón 'Siguiente' o llegamos al final.

¡Éxito! Se han extraído 48 laptops en total y se guardaron en 'laptops_mercadolibre_dataset.csv'

Las 5 laptops más baratas encontradas son:


,Modelo,Precio ($),Enlace
44,"Notebook HP 255 G10 con AMD Ryzen 3 7335U, 16 ...",4699,https://www.mercadolibre.com.mx/notebook-hp-25...
43,Consola Gamer Legion Go 8 Amd Ryzen Z2 32gb Ra...,4888,https://www.mercadolibre.com.mx/consola-gamer-...
39,"Laptop Hp 255 G10 15.6 Fhd Amd Ryzen 3 7335u, ...",4978,https://www.mercadolibre.com.mx/laptop-hp-255-...
47,Laptop HP 14 14-dq6101dx Intel N150 4 GB RAM 1...,5031,https://www.mercadolibre.com.mx/laptop-hp-14-1...
35,Laptop Acer Aspire 3 Amd Ryzen 7 32gb Ram 512g...,5789,https://www.mercadolibre.com.mx/laptop-acer-as...


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# 1. Configuración del encabezado (Disfraz de Googlebot)
headers = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

lista_laptops = []
paginas_a_extraer = 10  # Escanea las primeras 10 páginas (~480 registros)

print(f"Iniciando extracción masiva de hasta {paginas_a_extraer} páginas...\n")

for pagina in range(paginas_a_extraer):
    offset = (pagina * 48) + 1
    if pagina == 0:
        url_actual = 'https://listado.mercadolibre.com.mx/laptops'
    else:
        url_actual = f'https://listado.mercadolibre.com.mx/laptops_Desde_{offset}_NoIndex_True'
        
    print(f"-> Escaneando página {pagina + 1} ({url_actual})...")
    
    try:
        response = requests.get(url_actual, headers=headers, timeout=20)
    except Exception as e:
        print(f"Error al conectar: {e}")
        break

    if response.status_code != 200 or "ingresa a tu cuenta" in response.text.lower():
        print("¡Alerta! Bloqueo o muro detectado. Deteniendo extracción.")
        break
        
    soup = BeautifulSoup(response.text, 'lxml')
    
    titulos = soup.find_all('a', class_='poly-component__title')
    precios = soup.find_all('div', class_='poly-component__price')
    
    if not titulos:
        print("No se encontraron más productos. Fin del catálogo.")
        break

    for titulo, precio in zip(titulos, precios):
        span_precio = precio.find('span', class_='andes-money-amount__fraction')
        precio_texto = span_precio.text.strip() if span_precio else "0"
        enlace = titulo.get('href', '')

        lista_laptops.append({
            'Modelo_Completo': titulo.text.strip(),
            'Precio ($)': precio_texto,
            'Disponible': True,  # Si está activo en el listado, hay stock para compra
            'Enlace': enlace
        })
        
    time.sleep(1.5)  # Pausa respetuosa de 1.5s entre peticiones

# 2. Creación del DataFrame en Pandas
df = pd.DataFrame(lista_laptops)

if not df.empty:
    # Convertir precio a número entero
    df['Precio ($)'] = df['Precio ($)'].str.replace(',', '', regex=False).astype(int)

    # 3. Parsing de Especificaciones (Regex Corregidos)
    
    # Extraer Marca (Sin paréntesis anidados para evitar el ValueError)
    marcas_patron = r'HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Alienware|GIGABYTE|ThinkPad|Victus|ROG|TUF|IdeaPad|Vivobook'
    df['Marca'] = df['Modelo_Completo'].str.extract(f'({marcas_patron})', flags=re.IGNORECASE, expand=False).fillna('Otra')
    
    # Extraer RAM (prioriza la primera mención de GB o patrones de memoria)
    df['RAM'] = df['Modelo_Completo'].str.extract(r'(\b\d+\s*GB\b(?:\s*RAM|\s*Memoria)?)', flags=re.IGNORECASE, expand=False).fillna('No especificado')
    
    # Extraer Almacenamiento (detecta valores típicos de disco duro/SSD: 128GB, 256GB, 512GB, 1TB, 2TB)
    df['Almacenamiento'] = df['Modelo_Completo'].str.extract(
        r'(\b(?:128|256|512|1024|1|2)\s*(?:GB|TB)\s*(?:SSD|HDD)?|\b\d+\s*(?:GB|TB)\s*(?:SSD|HDD))', 
        flags=re.IGNORECASE, expand=False
    ).fillna('No especificado')
    
    # Extraer Procesador
    df['Procesador'] = df['Modelo_Completo'].str.extract(
        r'(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|M\d\s*Max|M\d\s*Pro|M\d\s*Ultra|Celeron|Athlon|Intel\s*N\d+)', 
        flags=re.IGNORECASE, expand=False
    ).fillna('No especificado')
    
    # Extraer Número de Núcleos (si se menciona explícitamente en el título)
    df['Nucleos'] = df['Modelo_Completo'].str.extract(r'(\d+\s*(?:núcleos|nucleos|cores|core\b))', flags=re.IGNORECASE, expand=False).fillna('No especificado')

    # Reordenar columnas para un CSV limpio
    columnas_ordenadas = ['Marca', 'Modelo_Completo', 'Precio ($)', 'RAM', 'Almacenamiento', 'Procesador', 'Nucleos', 'Disponible', 'Enlace']
    df = df[columnas_ordenadas]

    # Guardar en CSV
    df.to_csv('laptops_mercadolibre_masivo.csv', index=False, encoding='utf-8')
    print(f"\n¡Proceso completado con éxito! Se extrajeron {len(df)} registros.")
    print("Archivo guardado como 'laptops_mercadolibre_masivo.csv'.")
    
    # Mostrar resumen en el Notebook
    display(df.head(10))
else:
    print("No se pudieron extraer datos.")

Iniciando extracción masiva de hasta 10 páginas...

-> Escaneando página 1 (https://listado.mercadolibre.com.mx/laptops)...
-> Escaneando página 2 (https://listado.mercadolibre.com.mx/laptops_Desde_49_NoIndex_True)...
-> Escaneando página 3 (https://listado.mercadolibre.com.mx/laptops_Desde_97_NoIndex_True)...
-> Escaneando página 4 (https://listado.mercadolibre.com.mx/laptops_Desde_145_NoIndex_True)...
-> Escaneando página 5 (https://listado.mercadolibre.com.mx/laptops_Desde_193_NoIndex_True)...
-> Escaneando página 6 (https://listado.mercadolibre.com.mx/laptops_Desde_241_NoIndex_True)...
-> Escaneando página 7 (https://listado.mercadolibre.com.mx/laptops_Desde_289_NoIndex_True)...
-> Escaneando página 8 (https://listado.mercadolibre.com.mx/laptops_Desde_337_NoIndex_True)...
-> Escaneando página 9 (https://listado.mercadolibre.com.mx/laptops_Desde_385_NoIndex_True)...
-> Escaneando página 10 (https://listado.mercadolibre.com.mx/laptops_Desde_433_NoIndex_True)...

¡Proceso completado c

,Marca,Modelo_Completo,Precio ($),RAM,Almacenamiento,Procesador,Nucleos,Disponible,Enlace
0,HP,Laptop HP 250 G10 Intel Core i3 N305 16GB 256G...,13999,16GB,256GB SSD,Core i3,No especificado,True,https://www.mercadolibre.com.mx/laptop-hp-250-...
1,Apple,Mac Mini M4 24gb (memoria Unificada) 512 gb Ap...,28999,24gb,512 gb,No especificado,No especificado,True,https://www.mercadolibre.com.mx/mac-mini-m4-24...
2,Otra,Ultrabook Intel N150 Machcreator 15a 15.6 Pulg...,8138,8GB RAM,512GB SSD,Intel N150,No especificado,True,https://www.mercadolibre.com.mx/ultrabook-inte...
3,HP,Laptop HP 245 AMD Ryzen 3 7330U 16GB 512GB SSD...,11999,16GB,512GB SSD,Ryzen 3,No especificado,True,https://www.mercadolibre.com.mx/laptop-hp-245-...
4,Asus,Laptop Asus Vivobook Intel Core I3 8gb Ram 128...,7999,8gb Ram,128gb Ssd,Core I3,No especificado,True,https://www.mercadolibre.com.mx/laptop-asus-vi...
5,Apple,"Apple 2026 Macbook Neo (de 13 Pulgadas, Chip A...",14999,8gb Memoria,256 gb,No especificado,6 Núcleos,True,https://www.mercadolibre.com.mx/apple-2026-mac...
6,IdeaPad,"Laptop IdeaPad Slim 3 15,3 Intel Core I5 24gb ...",23499,24gb Ram,1tb Ssd,Core I5,No especificado,True,https://www.mercadolibre.com.mx/laptop-ideapad...
7,ASUS,Laptop ASUS Vivobook 15 Intel Core i3 1315U 24...,13499,24GB,24GB SSD,Core i3,No especificado,True,https://www.mercadolibre.com.mx/laptop-asus-vi...
8,Hp,Laptop Hp 14 Intel N150 4gb Ram 128gb Ssd Wind...,6999,4gb Ram,128gb Ssd,Intel N150,No especificado,True,https://www.mercadolibre.com.mx/laptop-hp-14-i...
9,Otra,Ordenador Portátil Intel Pentium Gold 16 Gb Ra...,5949,16 Gb Ram,No especificado,No especificado,No especificado,True,https://www.mercadolibre.com.mx/ordenador-port...


In [14]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

CARPETA_IMAGENES = 'laptop_images'
if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

# Headers más completos para evitar bloqueos
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'es-MX,es;q=0.9,en-US;q=0.8,en;q=0.7',
}

def limpiar_precio(precio_str):
    if not precio_str:
        return None
    limpio = re.sub(r'[^\d.]', '', precio_str.replace(',', ''))
    try:
        return float(limpio)
    except ValueError:
        return None

def estandarizar_ram(ram_str):
    if not ram_str:
        return None
    match = re.search(r'(\d+)\s*(gb|mb)', ram_str.lower())
    if match:
        valor, unidad = int(match.group(1)), match.group(2)
        return valor if unidad == 'gb' else int(valor / 1024)
    return None

def estandarizar_almacenamiento(str_almacenamiento):
    if not str_almacenamiento:
        return None, "Desconocido"
    texto = str_almacenamiento.lower()
    tipo = "SSD" if "ssd" in texto or "nvme" in texto else ("HDD" if "hdd" in texto or "disco duro" in texto else "SSD")
    match = re.search(r'(\d+)\s*(tb|gb)', texto)
    if match:
        valor, unidad = int(match.group(1)), match.group(2)
        capacidad_gb = valor * 1024 if unidad == 'tb' else valor
        return capacidad_gb, tipo
    return None, tipo

def detectar_gpu_dedicada(gpu_str, titulo_str):
    texto = f"{gpu_str} {titulo_str}".lower()
    gpus_dedicadas = ['rtx', 'gtx', 'radeon rx', 'quadro', 'geforce mx', 'arc a']
    return 1 if any(gpu in texto for gpu in gpus_dedicadas) else 0

def scrapear_inventario_laptops(url_busqueda, max_productos=20):
    laptops_lista = []
    
    response = requests.get(url_busqueda, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Buscamos elementos li o div genéricos que contienen cada tarjeta de producto
    items = soup.find_all('li', class_=re.compile('ui-search-layout__item'))
    if not items:
        # Selector alternativo si cambia la estructura de la cuadrícula
        items = soup.find_all('div', class_=re.compile('ui-search-result__wrapper'))

    print(f"Productos detectados en la página: {len(items)}")

    for idx, item in enumerate(items[:max_productos]):
        try:
            # Buscar título por etiqueta h2 o h3
            titulo_elem = item.find(re.compile('h[2-3]'))
            titulo = titulo_elem.text.strip() if titulo_elem else "Sin título"
            
            # Buscar precio
            precio_elem = item.find('span', class_='andes-money-amount__fraction')
            precio_raw = precio_elem.text if precio_elem else "0"
            precio_num = limpiar_precio(precio_raw)
            
            # Buscar enlace de la publicación
            url_elem = item.find('a', href=True)
            url_producto = url_elem['href'] if url_elem else ""
            
            # Buscar etiqueta de imagen
            img_elem = item.find('img')
            img_url = ""
            if img_elem:
                img_url = img_elem.get('data-src') or img_elem.get('src') or ""
            
            # Descargar imagen local
            ruta_img_local = ""
            if img_url and img_url.startswith('http'):
                ruta_img_local = os.path.join(CARPETA_IMAGENES, f"laptop_{idx:03d}.jpg")
                try:
                    img_data = requests.get(img_url, headers=HEADERS, timeout=5).content
                    with open(ruta_img_local, 'wb') as f:
                        f.write(img_data)
                except Exception as img_err:
                    print(f"Error descargando imagen {idx}: {img_err}")

            # Procesamiento de especificaciones
            ram_gb = estandarizar_ram(titulo) or 8
            almacenamiento_gb, tipo_almacenamiento = estandarizar_almacenamiento(titulo)
            if not almacenamiento_gb:
                almacenamiento_gb = 512
                
            gpu_dedicada = detectar_gpu_dedicada("", titulo)
            
            laptops_lista.append({
                'ID_Producto': f"LAP-{idx:03d}",
                'Nombre': titulo,
                'Marca': titulo.split()[0] if titulo else "Genérica",
                'Precio_MXN': precio_num,
                'RAM_GB': ram_gb,
                'Almacenamiento_GB': almacenamiento_gb,
                'Tipo_Almacenamiento': tipo_almacenamiento,
                'GPU_Dedicada': gpu_dedicada,
                'Ruta_Imagen_Local': ruta_img_local,
                'URL_Producto': url_producto
            })
            
        except Exception as e:
            print(f"Error procesando producto {idx}: {e}")
            continue

    # Si la lista sigue vacía, creamos un DataFrame con las columnas explícitas para evitar que truene el script
    columnas = ['ID_Producto', 'Nombre', 'Marca', 'Precio_MXN', 'RAM_GB', 
                'Almacenamiento_GB', 'Tipo_Almacenamiento', 'GPU_Dedicada', 
                'Ruta_Imagen_Local', 'URL_Producto']
    
    return pd.DataFrame(laptops_lista, columns=columnas)

if __name__ == "__main__":
    URL_OBJETIVO = 'https://listado.mercadolibre.com.mx/laptops'
    df_laptops = scrapear_inventario_laptops(URL_OBJETIVO, max_productos=15)
    
    if df_laptops.empty:
        print("\n[!] Advertencia: No se pudieron extraer datos. Revisa la conexión o el bloqueo de la página.")
    else:
        df_laptops.to_csv('laptops_base_limpia.csv', index=False)
        print("\nScraping completado con éxito. Muestra de datos:")
        print(df_laptops[['ID_Producto', 'Nombre', 'Precio_MXN', 'RAM_GB', 'Almacenamiento_GB', 'GPU_Dedicada', 'Ruta_Imagen_Local']].head())

Productos detectados en la página: 0

[!] Advertencia: No se pudieron extraer datos. Revisa la conexión o el bloqueo de la página.


In [16]:
# 1. Instalar la librería de Python
!pip install playwright

# 2. Descargar e instalar el navegador Chromium para Playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 14.0 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 19.7 MB/s eta 0:00:00
BEWARE: your OS is not officially supported by Playwright; downloading fallback build for ubuntu24.04-x64.
Error: Request to https://cdn.playwright.dev/builds/cft/153.0.8010.12/linux64/chrome-linux64.zip timed out after 30000ms
    at ClientRequest.<anonymous> (/home/solaris/Documents/proyectos/claralap/claralap/lib/python3.12/site-packages/playwright/driver/package/lib/coreBundle.js:8793:15)
    at ClientRequest.wrapper (node:events:639:12)
    at ClientRequest.emit (node:events:514:28)
    at TLSSocket.emitRequestTimeout (node:_http_client:1122:9)
    at TLSSocket.wrapper (node:events:639:12)
    at TLSSocket.emit (node:events:526:24)
    at Socket._onTimeout (node:net:824:8)
    at Timeout.onStreamTimeout (node:internal/stream_base_commons:237:10)
    at listOnTimeout (node:internal/timers:687:11)
    at p

In [21]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

CARPETA_IMAGENES = 'laptop_images'
if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

# Encabezado clave: Simular Googlebot para recibir HTML estático sin bloqueos JS
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'es-MX,es;q=0.9'
}

def limpiar_precio(precio_str):
    if not precio_str:
        return None
    limpio = re.sub(r'[^\d.]', '', precio_str.replace(',', ''))
    try:
        return float(limpio)
    except ValueError:
        return None

def estandarizar_ram(ram_str):
    if not ram_str:
        return 8
    match = re.search(r'(\d+)\s*(gb|mb)', ram_str.lower())
    if match:
        valor, unidad = int(match.group(1)), match.group(2)
        return valor if unidad == 'gb' else int(valor / 1024)
    return 8

def estandarizar_almacenamiento(str_almacenamiento):
    if not str_almacenamiento:
        return 512, "SSD"
    texto = str_almacenamiento.lower()
    tipo = "SSD" if "ssd" in texto or "nvme" in texto else ("HDD" if "hdd" in texto or "disco duro" in texto else "SSD")
    match = re.search(r'(\d+)\s*(tb|gb)', texto)
    if match:
        valor, unidad = int(match.group(1)), match.group(2)
        capacidad_gb = valor * 1024 if unidad == 'tb' else valor
        return capacidad_gb, tipo
    return 512, tipo

def detectar_gpu_dedicada(titulo_str):
    texto = titulo_str.lower()
    gpus_dedicadas = ['rtx', 'gtx', 'radeon rx', 'quadro', 'geforce mx', 'arc a']
    return 1 if any(gpu in texto for gpu in gpus_dedicadas) else 0

def scrapear_mercadolibre_googlebot(max_productos=20):
    laptops_lista = []
    url_objetivo = 'https://listado.mercadolibre.com.mx/laptops'
    
    print("Conectando a Mercado Libre mediante bypass de Googlebot...")
    try:
        response = requests.get(url_objetivo, headers=HEADERS, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"Error en la petición: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Selectores para el diseño moderno de tarjetas (poly-card y contenedores tradicionales)
    tarjetas = soup.find_all(['div', 'li'], class_=re.compile(r'poly-card|ui-search-layout__item'))
    print(f"Tarjetas de productos detectadas: {len(tarjetas)}")

    for idx, item in enumerate(tarjetas[:max_productos]):
        try:
            # Título y Enlace
            titulo_elem = item.find('a', class_=re.compile(r'poly-component__title|ui-search-item__title')) or item.find(re.compile(r'h[2-3]'))
            titulo = titulo_elem.text.strip() if titulo_elem else "Sin título"
            url_producto = titulo_elem['href'] if titulo_elem and titulo_elem.has_attr('href') else ""

            # Precio
            precio_elem = item.find('span', class_='andes-money-amount__fraction')
            precio_raw = precio_elem.text.strip() if precio_elem else "0"
            precio_num = limpiar_precio(precio_raw)

            # Imagen (Extraer src o data-src)
            img_elem = item.find('img')
            img_url = ""
            if img_elem:
                img_url = img_elem.get('data-src') or img_elem.get('src') or ""

            # Descarga local de imagen
            ruta_img_local = ""
            if img_url and img_url.startswith('http'):
                ruta_img_local = os.path.join(CARPETA_IMAGENES, f"laptop_{idx:03d}.jpg")
                try:
                    img_data = requests.get(img_url, timeout=5).content
                    with open(ruta_img_local, 'wb') as f:
                        f.write(img_data)
                except Exception:
                    ruta_img_local = "Error en descarga"

            # Parseo de especificaciones mediante Regex
            ram_gb = estandarizar_ram(titulo)
            almacenamiento_gb, tipo_almacenamiento = estandarizar_almacenamiento(titulo)
            gpu_dedicada = detectar_gpu_dedicada(titulo)

            laptops_lista.append({
                'ID_Producto': f"LAP-{idx:03d}",
                'Nombre': titulo,
                'Marca': titulo.split()[0] if titulo else "Genérica",
                'Precio_MXN': precio_num,
                'RAM_GB': ram_gb,
                'Almacenamiento_GB': almacenamiento_gb,
                'Tipo_Almacenamiento': tipo_almacenamiento,
                'GPU_Dedicada': gpu_dedicada,
                'Ruta_Imagen_Local': ruta_img_local,
                'URL_Producto': url_producto
            })
        except Exception as e:
            print(f"Error analizando tarjeta {idx}: {e}")
            continue

    columnas = ['ID_Producto', 'Nombre', 'Marca', 'Precio_MXN', 'RAM_GB', 
                'Almacenamiento_GB', 'Tipo_Almacenamiento', 'GPU_Dedicada', 
                'Ruta_Imagen_Local', 'URL_Producto']
    
    return pd.DataFrame(laptops_lista, columns=columnas)

# Ejecución
df_laptops = scrapear_mercadolibre_googlebot(max_productos=20)

if not df_laptops.empty:
    df_laptops.to_csv('laptops_base_limpia.csv', index=False, encoding='utf-8')
    print("\nDataset generado con éxito:")
    display(df_laptops[['ID_Producto', 'Nombre', 'Precio_MXN', 'RAM_GB', 'Almacenamiento_GB', 'GPU_Dedicada', 'Ruta_Imagen_Local']].head())
else:
    print("\nNo se pudieron extraer los productos.")

Conectando a Mercado Libre mediante bypass de Googlebot...


/home/solaris/Documents/proyectos/claralap/claralap/lib/python3.12/site-packages/bs4/builder/__init__.py:411: RuntimeWarning: coroutine 'scrapear_con_playwright' was never awaited
  for attr in list(modified_attrs.keys()):


Tarjetas de productos detectadas: 193

Dataset generado con éxito:


,ID_Producto,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,GPU_Dedicada,Ruta_Imagen_Local
0,LAP-000,Laptop HP 250 G10 Intel Core i3 N305 16GB 256G...,13999.0,16,16,0,laptop_images/laptop_000.jpg
1,LAP-001,Laptop HP 250 G10 Intel Core i3 N305 16GB 256G...,13999.0,16,16,0,laptop_images/laptop_001.jpg
2,LAP-002,Sin título,0.0,8,512,0,laptop_images/laptop_002.jpg
3,LAP-003,Laptop HP 250 G10 Intel Core i3 N305 16GB 256G...,13999.0,16,16,0,
4,LAP-004,Mac Mini M4 24gb (memoria Unificada) 512 gb Ap...,28999.0,24,24,0,laptop_images/laptop_004.jpg


In [22]:
import re
import numpy as np
import pandas as pd

# 1. Cargar la base raspada
df = pd.read_csv('laptops_base_limpia.csv')

# 2. Eliminar basura (registros sin título o sin precio válido)
df_clean = df[(df['Nombre'] != 'Sin título') & (df['Precio_MXN'] > 0)].copy()

# 3. Filtrar solo laptops (excluir Mini PCs como Mac Mini)
df_clean = df_clean[~df_clean['Nombre'].str.contains('Mac Mini', case=False)].copy()

# 4. Extracción corregida de Marca Real
marcas_patron = r'(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Machcreator|Gigabyte)'
df_clean['Marca_Real'] = (
    df_clean['Nombre']
    .str.extract(marcas_patron, flags=re.IGNORECASE, expand=False)
    .fillna('Otra')
)
df_clean['Marca_Real'] = df_clean['Marca_Real'].replace(
    {'MacBook': 'Apple', 'Machcreator': 'Intel Machcreator'}
)

# 5. Extracción de Familia de Procesador
cpu_patron = r'(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|Intel\s*N\d+|Celeron|Athlon)'
df_clean['Procesador'] = (
    df_clean['Nombre']
    .str.extract(cpu_patron, flags=re.IGNORECASE, expand=False)
    .fillna('Gama Entrada / No especificado')
)

# Mapeo de Gama de CPU (Ordinal para Modelos de ML)
gama_cpu_map = {
    'Intel N150': 1,
    'Celeron': 1,
    'Core i3': 2,
    'Core I3': 2,
    'Ryzen 3': 2,
    'Core i5': 3,
    'Ryzen 5': 3,
    'Core i7': 4,
    'Ryzen 7': 4,
    'Apple M4': 5,
}
df_clean['Procesador_Gama'] = (
    df_clean['Procesador'].map(gama_cpu_map).fillna(2).astype(int)
)

# 6. Extracción de Tamaño de Pantalla
df_clean['Pantalla_Pulgadas'] = (
    df_clean['Nombre']
    .str.extract(r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', flags=re.IGNORECASE, expand=False)
    .fillna('15.6')
    .astype(float)
)

# 7. Corrección de Almacenamiento SSD en GB (extrayendo explícitamente valores de 128, 256, 512, 1000/1TB)
df_clean['Almacenamiento_GB_Real'] = (
    df_clean['Nombre']
    .str.extract(
        r'(\d{3,4})\s*(?:GB|TB)\s*(?:SSD|HDD)|(?:SSD|HDD)\s*(\d{3,4})',
        flags=re.IGNORECASE,
    )
    .bfill(axis=1)[0]
    .fillna(256)
    .astype(int)
)

# 8. Creación de Índice Numérico de Hardware (Hardware Score)
df_clean['Hardware_Score'] = (
    (df_clean['RAM_GB'] * 0.3)
    + (df_clean['Procesador_Gama'] * 3.0)
    + (df_clean['GPU_Dedicada'] * 5.0)
    + (df_clean['Almacenamiento_GB_Real'] / 100 * 0.5)
)

# 9. Guardar la nueva base estandarizada
df_clean.to_csv('laptops_base_enriquecida.csv', index=False)
print(
    'Base de datos enriquecida guardada exitosamente. Filas válidas:',
    len(df_clean),
)

Base de datos enriquecida guardada exitosamente. Filas válidas: 11


In [23]:
import os
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

CARPETA_IMAGENES = "laptop_images"
if not os.path.exists(CARPETA_IMAGENES):
    os.makedirs(CARPETA_IMAGENES)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; Googlebot/2.1;"
        " +http://www.google.com/bot.html)"
    ),
    "Accept-Language": "es-MX,es;q=0.9",
}


def limpiar_precio(precio_str):
    if not precio_str:
        return None
    limpio = re.sub(r"[^\d.]", "", precio_str.replace(",", ""))
    try:
        return float(limpio)
    except ValueError:
        return None


def extraer_ram(titulo):
    match = re.search(
        r"(\d+)\s*(gb|mb)\s*(?:ram|memoria)?", titulo, re.IGNORECASE
    )
    if match:
        valor, unidad = int(match.group(1)), match.group(2).lower()
        return valor if unidad == "gb" else int(valor / 1024)
    return None


def extraer_almacenamiento(titulo):
    match = re.search(
        r"(\d+)\s*(tb|gb)\s*(ssd|hdd|nvme)?|(ssd|hdd|nvme)\s*(\d+)\s*(tb|gb)",
        titulo,
        re.IGNORECASE,
    )
    if match:
        groups = [g for g in match.groups() if g is not None]
        num = next((int(g) for g in groups if g.isdigit()), None)
        unit = next(
            (g.lower() for g in groups if g.lower() in ["gb", "tb"]), "gb"
        )
        tipo = next(
            (g.upper() for g in groups if g.upper() in ["SSD", "HDD", "NVME"]),
            "SSD",
        )
        if num:
            capacidad_gb = num * 1024 if unit == "tb" else num
            return capacidad_gb, tipo
    return None, "SSD"


def extraer_marca(titulo):
    marcas_patron = r"(HP|Lenovo|Asus|Dell|Apple|MacBook|Acer|MSI|Huawei|Samsung|Gigabyte|Machcreator)"
    match = re.search(marcas_patron, titulo, re.IGNORECASE)
    if match:
        marca = match.group(1)
        if marca.lower() == "macbook":
            return "Apple"
        return marca.capitalize()
    return "Otra"


def extraer_procesador(titulo):
    match = re.search(
        r"(Core\s*i\d|Ryzen\s*\d|Apple\s*M\d|Intel\s*N\d+|Celeron|Athlon)",
        titulo,
        re.IGNORECASE,
    )
    return match.group(1).title() if match else "No especificado"


def extraer_pantalla(titulo):
    match = re.search(
        r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”)', titulo, re.IGNORECASE
    )
    return float(match.group(1)) if match else None


def detectar_gpu_dedicada(titulo):
    gpus = ["rtx", "gtx", "radeon rx", "quadro", "geforce mx", "arc a"]
    return 1 if any(gpu in titulo.lower() for gpu in gpus) else 0


def scrapear_mercadolibre_explicito(max_productos=20):
    laptops_lista = []
    url_objetivo = "https://listado.mercadolibre.com.mx/laptops"

    response = requests.get(url_objetivo, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
    tarjetas = soup.find_all(
        ["div", "li"], class_=re.compile(r"poly-card|ui-search-layout__item")
    )

    idx = 0
    for item in tarjetas:
        if idx >= max_productos:
            break

        titulo_elem = item.find(
            "a", class_=re.compile(r"poly-component__title|ui-search-item__title")
        ) or item.find(re.compile(r"h[2-3]"))
        titulo = titulo_elem.text.strip() if titulo_elem else ""

        # Omitir tarjetas sin título o productos que no sean laptops
        if not titulo or titulo == "Sin título" or "mac mini" in titulo.lower():
            continue

        precio_elem = item.find("span", class_="andes-money-amount__fraction")
        precio_num = (
            limpiar_precio(precio_elem.text.strip()) if precio_elem else None
        )

        if not precio_num or precio_num == 0:
            continue

        url_producto = (
            titulo_elem["href"]
            if titulo_elem and titulo_elem.has_attr("href")
            else ""
        )

        img_elem = item.find("img")
        img_url = ""
        if img_elem:
            img_url = img_elem.get("data-src") or img_elem.get("src") or ""

        ruta_img_local = ""
        if img_url and img_url.startswith("http"):
            ruta_img_local = os.path.join(
                CARPETA_IMAGENES, f"laptop_{idx:03d}.jpg"
            )
            try:
                img_data = requests.get(img_url, timeout=5).content
                with open(ruta_img_local, "wb") as f:
                    f.write(img_data)
            except Exception:
                ruta_img_local = ""

        ram_gb = extraer_ram(titulo)
        almacenamiento_gb, tipo_almacenamiento = extraer_almacenamiento(titulo)
        marca = extraer_marca(titulo)
        procesador = extraer_procesador(titulo)
        pantalla = extraer_pantalla(titulo)
        gpu_dedicada = detectar_gpu_dedicada(titulo)

        laptops_lista.append({
            "ID_Producto": f"LAP-{idx:03d}",
            "Nombre": titulo,
            "Marca": marca,
            "Precio_MXN": precio_num,
            "RAM_GB": ram_gb,
            "Almacenamiento_GB": almacenamiento_gb,
            "Tipo_Almacenamiento": tipo_almacenamiento,
            "Procesador": procesador,
            "Pantalla_Pulgadas": pantalla,
            "GPU_Dedicada": gpu_dedicada,
            "Ruta_Imagen_Local": ruta_img_local,
            "URL_Producto": url_producto,
        })
        idx += 1

    df_resultado = pd.DataFrame(laptops_lista)
    df_resultado.to_csv("laptops_base_explicita.csv", index=False)
    return df_resultado


# Ejecutar scraping
if __name__ == "__main__":
    df_final = scrapear_mercadolibre_explicito(max_productos=20)
    print("Scraping finalizado. Columnas generadas:")
    print(df_final.columns.tolist())

Scraping finalizado. Columnas generadas:
['ID_Producto', 'Nombre', 'Marca', 'Precio_MXN', 'RAM_GB', 'Almacenamiento_GB', 'Tipo_Almacenamiento', 'Procesador', 'Pantalla_Pulgadas', 'GPU_Dedicada', 'Ruta_Imagen_Local', 'URL_Producto']


In [24]:
import re
import time
import urllib.parse
from bs4 import BeautifulSoup
import pandas as pd
import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


def aislar_modelo_clave(nombre_completo):
    """Elimina frases publicitarias y conserva solo Marca + Serie + Procesador."""
    ruido = (
        r"teclado español|mouse dxt|ideal para estudiante|y oficina|portátil"
        r" ligero|distribuidor autorizado|envío gratis"
    )
    limpio = re.sub(ruido, "", nombre_completo, flags=re.IGNORECASE)
    # Conserva los primeros 6-7 términos más relevantes del título
    palabras = limpio.split()[:7]
    return " ".join(palabras)


def buscar_snippet_ddg_lite(query_modelo):
    """Consulta DuckDuckGo Lite para obtener resultados en HTML estático puro."""
    url = "https://lite.duckduckgo.com/lite/"
    payload = {"q": f"{query_modelo} especificaciones tecnicas"}

    try:
        response = requests.post(url, data=payload, headers=HEADERS, timeout=8)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            # Extraer todos los fragmentos (snippets) del texto de resultados
            snippets = [
                td.text.strip()
                for td in soup.find_all("td", class_="result-snippet")
            ]
            return " ".join(snippets)
    except Exception:
        pass
    return ""


def extraer_datos_de_texto(texto_resultado):
    """Analiza el texto recuperado de la web mediante expresiones regulares."""
    datos = {
        "Pantalla_Pulgadas": None,
        "Numero_Nucleos": None,
        "Generacion_CPU": None,
    }

    if not texto_resultado:
        return datos

    # 1. Buscar Pantalla (ej: 14", 15.6 pulgadas)
    match_pantalla = re.search(
        r'(\d{2}(?:\.\d)?)\s*(?:"|pulgadas|”|fhd|hd)',
        texto_resultado,
        re.IGNORECASE,
    )
    if match_pantalla:
        datos["Pantalla_Pulgadas"] = float(match_pantalla.group(1))

    # 2. Buscar Núcleos (ej: 8 núcleos, 12 cores, quad-core)
    match_nucleos = re.search(
        r"(\d+)\s*(?:núcleos|nucleos|cores|core\b)",
        texto_resultado,
        re.IGNORECASE,
    )
    if match_nucleos:
        datos["Numero_Nucleos"] = int(match_nucleos.group(1))

    # 3. Buscar Generación (ej: 12ª generación, 13th gen, Zen 3)
    match_gen = re.search(
        r"(\d{1,2})ª?\s*(?:generación|generacion|gen)|zen\s*\d",
        texto_resultado,
        re.IGNORECASE,
    )
    if match_gen:
        datos["Generacion_CPU"] = match_gen.group(0).title()

    return datos


def enriquecer_desde_web_abierta(
    csv_entrada="laptops_base_explicita.csv",
    csv_salida="laptops_base_enriquecida.csv",
):
    df = pd.read_csv(csv_entrada)

    # Asegurar la existencia de las nuevas columnas
    for col in ["Numero_Nucleos", "Generacion_CPU"]:
        if col not in df.columns:
            df[col] = None

    print(f"Iniciando enriquecimiento externo para {len(df)} registros...\n")

    for idx, row in df.iterrows():
        # Consultar solo si faltan datos en la fila actual
        falta_pantalla = pd.isna(row.get("Pantalla_Pulgadas"))
        falta_nucleos = pd.isna(row.get("Numero_Nucleos"))

        if falta_pantalla or falta_nucleos:
            modelo_limpio = aislar_modelo_clave(str(row["Nombre"]))
            print(f"[{idx+1}/{len(df)}] Buscando especificaciones de: '{modelo_limpio}'...")

            snippet_texto = buscar_snippet_ddg_lite(modelo_limpio)
            specs = extraer_datos_de_texto(snippet_texto)

            if falta_pantalla and specs["Pantalla_Pulgadas"]:
                df.at[idx, "Pantalla_Pulgadas"] = specs["Pantalla_Pulgadas"]

            if specs["Numero_Nucleos"]:
                df.at[idx, "Numero_Nucleos"] = specs["Numero_Nucleos"]

            if specs["Generacion_CPU"]:
                df.at[idx, "Generacion_CPU"] = specs["Generacion_CPU"]

            time.sleep(1.5)  # Pausa respetuosa entre búsquedas

    # Imputación de valores por defecto en caso de no ser encontrados
    df["Pantalla_Pulgadas"] = df["Pantalla_Pulgadas"].fillna(15.6)
    df["Numero_Nucleos"] = df["Numero_Nucleos"].fillna("No especificado")
    df["Generacion_CPU"] = df["Generacion_CPU"].fillna("No especificado")

    df.to_csv(csv_salida, index=False)
    print(f"\n Dataset enriquecido exitosamente y guardado en '{csv_salida}'.")
    return df


if __name__ == "__main__":
    df_resultado = enriquecer_desde_web_abierta()

Iniciando enriquecimiento externo para 20 registros...

[1/20] Buscando especificaciones de: 'Laptop HP 250 G10 Intel Core i3'...
[2/20] Buscando especificaciones de: 'Laptop HP 250 G10 Intel Core i3'...
[3/20] Buscando especificaciones de: 'Laptop HP 250 G10 Intel Core i3'...
[4/20] Buscando especificaciones de: 'Ultrabook Intel N150 Machcreator 15a 15.6 Pulgadas'...
[5/20] Buscando especificaciones de: 'Ultrabook Intel N150 Machcreator 15a 15.6 Pulgadas'...
[6/20] Buscando especificaciones de: 'Ultrabook Intel N150 Machcreator 15a 15.6 Pulgadas'...
[7/20] Buscando especificaciones de: 'Laptop HP 245 AMD Ryzen 3 7330U'...
[8/20] Buscando especificaciones de: 'Laptop HP 245 AMD Ryzen 3 7330U'...
[9/20] Buscando especificaciones de: 'Laptop HP 245 AMD Ryzen 3 7330U'...
[10/20] Buscando especificaciones de: 'Laptop Asus Vivobook Intel Core I3 8gb'...
[11/20] Buscando especificaciones de: 'Laptop Asus Vivobook Intel Core I3 8gb'...
[12/20] Buscando especificaciones de: 'Laptop Asus Vivobo

In [25]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

# 1. BASE DE DATOS DETERMINISTA DE PROCESADORES FRECUENTES
CPU_DATABASE = {
    # Intel N-Series
    "INTEL N100": {"nucleos": 4, "gen": "Alder Lake-N", "pulgadas": None},
    "INTEL N150": {"nucleos": 4, "gen": "Twin Lake (2024)", "pulgadas": None},
    "INTEL N200": {"nucleos": 4, "gen": "Alder Lake-N", "pulgadas": None},
    # Intel Core i3 / i5 / i7
    "I3-N305": {"nucleos": 8, "gen": "12ª Generación", "pulgadas": None},
    "I3-1215U": {"nucleos": 6, "gen": "12ª Generación", "pulgadas": None},
    "I3-1315U": {"nucleos": 6, "gen": "13ª Generación", "pulgadas": None},
    "I5-1235U": {"nucleos": 10, "gen": "12ª Generación", "pulgadas": None},
    "I5-1335U": {"nucleos": 10, "gen": "13ª Generación", "pulgadas": None},
    "I7-1255U": {"nucleos": 10, "gen": "12ª Generación", "pulgadas": None},
    "I7-1355U": {"nucleos": 10, "gen": "13ª Generación", "pulgadas": None},
    # AMD Ryzen
    "RYZEN 3 7330U": {"nucleos": 4, "gen": "Serie 7000 (Zen 3)", "pulgadas": None},
    "RYZEN 5 5500U": {"nucleos": 6, "gen": "Serie 5000 (Zen 2)", "pulgadas": None},
    "RYZEN 5 7520U": {"nucleos": 4, "gen": "Serie 7000 (Zen 2)", "pulgadas": None},
    "RYZEN 7 5700U": {"nucleos": 8, "gen": "Serie 5000 (Zen 2)", "pulgadas": None},
    "RYZEN 7 7730U": {"nucleos": 8, "gen": "Serie 7000 (Zen 3)", "pulgadas": None},
    # Apple Silicon
    "APPLE M1": {"nucleos": 8, "gen": "1ª Generación Apple", "pulgadas": 13.3},
    "APPLE M2": {"nucleos": 8, "gen": "2ª Generación Apple", "pulgadas": 13.6},
    "APPLE M3": {"nucleos": 8, "gen": "3ª Generación Apple", "pulgadas": 14.2},
    "APPLE M4": {"nucleos": 10, "gen": "4ª Generación Apple", "pulgadas": 14.2},
}


def extraer_clave_cpu(titulo):
    """Extrae el código exacto de procesador desde el título."""
    patterns = [
        r"(i[3579]\s*[-_]?\s*(?:\d{4,5}[A-Z]*|N\d{3}))",  # ej: i3 N305, i5-1235U
        r"(Intel\s*N\d{3})",                             # ej: Intel N150
        r"(Ryzen\s*[3579]\s*\d{4}[A-Z]*)",               # ej: Ryzen 3 7330U
        r"(Apple\s*M\d\s*(?:Pro|Max)?)",                 # ej: Apple M4
    ]
    for p in patterns:
        match = re.search(p, str(titulo), re.IGNORECASE)
        if match:
            return match.group(1).upper().replace(" ", "-").replace("INTEL-", "INTEL ")
    return None


def buscar_specs_web_focalizada(cpu_code):
    """Realiza una búsqueda web dirigida exclusivamente al código del procesador."""
    if not cpu_code:
        return {}

    url = "https://lite.duckduckgo.com/lite/"
    payload = {"q": f"{cpu_code} processor specs cores generation"}

    try:
        response = requests.post(url, data=payload, headers=HEADERS, timeout=6)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            snippets = " ".join([td.text for td in soup.find_all("td", class_="result-snippet")])

            specs = {}
            # Regex multilingüe para núcleos
            match_cores = re.search(r"(\d+)\s*(?:cores|core|núcleos|nucleos|octa-core|quad-core)", snippets, re.IGNORECASE)
            if match_cores:
                specs["nucleos"] = int(match_cores.group(1))

            # Regex para generación/arquitectura
            match_gen = re.search(r"(\d{1,2}(?:th|rd|st|ª)?\s*gen(?:eration)?|zen\s*\d)", snippets, re.IGNORECASE)
            if match_gen:
                specs["gen"] = match_gen.group(0).title()

            return specs
    except Exception:
        pass
    return {}


def enriquecer_dataset_optimizado(csv_entrada="laptops_base_explicita.csv", csv_salida="laptops_base_enriquecida.csv"):
    df = pd.read_csv(csv_entrada)

    for col in ["Numero_Nucleos", "Generacion_CPU"]:
        if col not in df.columns:
            df[col] = None

    print(f"Iniciando enriquecimiento optimizado para {len(df)} registros...\n")

    for idx, row in df.iterrows():
        titulo = str(row["Nombre"])
        clave_cpu = extraer_clave_cpu(titulo)

        nucleos_encontrados = None
        gen_encontrada = None

        # PASO 1: Consulta en Base de Datos Interna (Instagráfica y 100% Precisa)
        if clave_cpu and clave_cpu in CPU_DATABASE:
            data = CPU_DATABASE[clave_cpu]
            nucleos_encontrados = data["nucleos"]
            gen_encontrada = data["gen"]
            print(f"[{idx+1}/{len(df)}] [Match BD] {clave_cpu} -> {nucleos_encontrados} Núcleos | {gen_encontrada}")

        # PASO 2: Búsqueda Focalizada Web (Fallback si no está en BD)
        elif clave_cpu:
            print(f"[{idx+1}/{len(df)}] [Búsqueda Web] Consultando procesador: {clave_cpu}...")
            specs_web = buscar_specs_web_focalizada(clave_cpu)
            nucleos_encontrados = specs_web.get("nucleos")
            gen_encontrada = specs_web.get("gen")
            time.sleep(1.0)

        # Asignar resultados extraídos
        if pd.isna(row.get("Numero_Nucleos")) and nucleos_encontrados:
            df.at[idx, "Numero_Nucleos"] = nucleos_encontrados

        if pd.isna(row.get("Generacion_CPU")) and gen_encontrada:
            df.at[idx, "Generacion_CPU"] = gen_encontrada

    # Valores por defecto para registros con títulos incompletos
    df["Pantalla_Pulgadas"] = df["Pantalla_Pulgadas"].fillna(15.6)
    df["Numero_Nucleos"] = df["Numero_Nucleos"].fillna(4)
    df["Generacion_CPU"] = df["Generacion_CPU"].fillna("No especificado")

    df.to_csv(csv_salida, index=False)
    print(f"\n Dataset enriquecido exitosamente en '{csv_salida}'.")
    return df


if __name__ == "__main__":
    enriquecer_dataset_optimizado()

Iniciando enriquecimiento optimizado para 20 registros...

[1/20] [Match BD] I3-N305 -> 8 Núcleos | 12ª Generación
[2/20] [Match BD] I3-N305 -> 8 Núcleos | 12ª Generación
[3/20] [Match BD] I3-N305 -> 8 Núcleos | 12ª Generación
[4/20] [Match BD] INTEL N150 -> 4 Núcleos | Twin Lake (2024)
[5/20] [Match BD] INTEL N150 -> 4 Núcleos | Twin Lake (2024)
[6/20] [Match BD] INTEL N150 -> 4 Núcleos | Twin Lake (2024)
[7/20] [Búsqueda Web] Consultando procesador: RYZEN-3-7330U...
[8/20] [Búsqueda Web] Consultando procesador: RYZEN-3-7330U...
[9/20] [Búsqueda Web] Consultando procesador: RYZEN-3-7330U...
[19/20] [Match BD] I3-1315U -> 6 Núcleos | 13ª Generación
[20/20] [Match BD] I3-1315U -> 6 Núcleos | 13ª Generación

 Dataset enriquecido exitosamente en 'laptops_base_enriquecida.csv'.


In [27]:
import re
import pandas as pd

# BASE DE DATOS TÉCNICA DE PROCESADORES (LOOKUP TABLE)
CATALOGO_CPU = [
    # Intel Core (N-Series y F-Series)
    {
        "patron": r"i3\s*[-_]?\s*N305",
        "modelo": "Intel Core i3-N305",
        "nucleos": 8,
        "gen": "12ª Gen (Alder Lake-N)",
    },
    {
        "patron": r"i3\s*[-_]?\s*1215U",
        "modelo": "Intel Core i3-1215U",
        "nucleos": 6,
        "gen": "12ª Gen (Alder Lake)",
    },
    {
        "patron": r"i3\s*[-_]?\s*1315U",
        "modelo": "Intel Core i3-1315U",
        "nucleos": 6,
        "gen": "13ª Gen (Raptor Lake)",
    },
    {
        "patron": r"i5\s*[-_]?\s*1235U",
        "modelo": "Intel Core i5-1235U",
        "nucleos": 10,
        "gen": "12ª Gen (Alder Lake)",
    },
    {
        "patron": r"i5\s*[-_]?\s*1335U",
        "modelo": "Intel Core i5-1335U",
        "nucleos": 10,
        "gen": "13ª Gen (Raptor Lake)",
    },
    {
        "patron": r"i7\s*[-_]?\s*1255U",
        "modelo": "Intel Core i7-1255U",
        "nucleos": 10,
        "gen": "12ª Gen (Alder Lake)",
    },
    {
        "patron": r"i7\s*[-_]?\s*1355U",
        "modelo": "Intel Core i7-1355U",
        "nucleos": 10,
        "gen": "13ª Gen (Raptor Lake)",
    },
    # Intel N-Series
    {
        "patron": r"N100",
        "modelo": "Intel N100",
        "nucleos": 4,
        "gen": "Alder Lake-N",
    },
    {
        "patron": r"N150",
        "modelo": "Intel N150",
        "nucleos": 4,
        "gen": "Twin Lake (2024)",
    },
    {
        "patron": r"N200",
        "modelo": "Intel N200",
        "nucleos": 4,
        "gen": "Alder Lake-N",
    },
    # AMD Ryzen
    {
        "patron": r"Ryzen\s*3\s*7330U",
        "modelo": "AMD Ryzen 3 7330U",
        "nucleos": 4,
        "gen": "Serie 7000 (Zen 3)",
    },
    {
        "patron": r"Ryzen\s*5\s*5500U",
        "modelo": "AMD Ryzen 5 5500U",
        "nucleos": 6,
        "gen": "Serie 5000 (Zen 2)",
    },
    {
        "patron": r"Ryzen\s*5\s*7520U",
        "modelo": "AMD Ryzen 5 7520U",
        "nucleos": 4,
        "gen": "Serie 7000 (Zen 2)",
    },
    {
        "patron": r"Ryzen\s*7\s*5700U",
        "modelo": "AMD Ryzen 7 5700U",
        "nucleos": 8,
        "gen": "Serie 5000 (Zen 2)",
    },
    {
        "patron": r"Ryzen\s*7\s*7730U",
        "modelo": "AMD Ryzen 7 7730U",
        "nucleos": 8,
        "gen": "Serie 7000 (Zen 3)",
    },
    # Apple Silicon
    {
        "patron": r"\bM1\b",
        "modelo": "Apple M1",
        "nucleos": 8,
        "gen": "1ª Gen Apple Silicon",
    },
    {
        "patron": r"\bM2\b",
        "modelo": "Apple M2",
        "nucleos": 8,
        "gen": "2ª Gen Apple Silicon",
    },
    {
        "patron": r"\bM3\b",
        "modelo": "Apple M3",
        "nucleos": 8,
        "gen": "3ª Gen Apple Silicon",
    },
    {
        "patron": r"\bM4\b",
        "modelo": "Apple M4",
        "nucleos": 10,
        "gen": "4ª Gen Apple Silicon",
    },
]


def extraer_marca_corregida(titulo):
    marcas = {
        r"\bHP\b": "HP",
        r"\bLenovo\b": "Lenovo",
        r"\bAsus\b|\bVivobook\b": "ASUS",
        r"\bDell\b": "Dell",
        r"\bApple\b|\bMacBook\b|\bMac\b": "Apple",
        r"\bAcer\b": "Acer",
        r"\bMSI\b": "MSI",
        r"\bHuawei\b|\bMateBook\b": "Huawei",
        r"\bSamsung\b": "Samsung",
        r"\bMachcreator\b": "Machcreator",
    }
    for patron, marca in marcas.items():
        if re.search(patron, titulo, re.IGNORECASE):
            return marca
    return "Otra"


def extraer_almacenamiento_corregido(titulo):
    # Buscar patrones específicos de SSD/HDD/NVMe (ej. 256GB SSD, 512 GB NVMe)
    match = re.search(
        r"(\d+)\s*(GB|TB)\s*(SSD|HDD|NVMe)", titulo, re.IGNORECASE
    )
    if match:
        val = int(match.group(1))
        unit = match.group(2).upper()
        tipo = match.group(3).upper()
        return (val * 1024 if unit == "TB" else val), tipo

    # Fallback: Buscar combinaciones de capacidad secundaria
    matches = re.findall(r"(\d+)\s*(?:GB|TB)", titulo, re.IGNORECASE)
    if len(matches) >= 2:
        val = int(matches[1])
        if val >= 128 or val == 1:
            return (val * 1024 if val == 1 else val), "SSD"

    return None, "SSD"


def identificar_procesador_completo(titulo):
    for cpu in CATALOGO_CPU:
        if re.search(cpu["patron"], titulo, re.IGNORECASE):
            return cpu["modelo"], cpu["nucleos"], cpu["gen"]

    # Fallback general para CPUs no catalogadas explícitamente
    if re.search(r"Core\s*i3", titulo, re.IGNORECASE):
        return "Intel Core i3", 4, "No especificado"
    if re.search(r"Core\s*i5", titulo, re.IGNORECASE):
        return "Intel Core i5", 6, "No especificado"
    if re.search(r"Core\s*i7", titulo, re.IGNORECASE):
        return "Intel Core i7", 8, "No especificado"
    if re.search(r"Ryzen\s*3", titulo, re.IGNORECASE):
        return "AMD Ryzen 3", 4, "No especificado"
    if re.search(r"Ryzen\s*5", titulo, re.IGNORECASE):
        return "AMD Ryzen 5", 6, "No especificado"

    return "No especificado", 4, "No especificado"


def enriquecer_base_datos(
    csv_entrada="laptops_base_limpia.csv",
    csv_salida="laptops_base_enriquecida.csv",
):
    df = pd.read_csv(csv_entrada)

    # Filtrar registros no válidos (ej. 'Sin título')
    df = df[df["Nombre"] != "Sin título"].copy()

    modelos_cpu, nucleos_list, gen_list = [], [], []
    marcas_list, alm_gb_list, alm_tipo_list = [], [], []

    for idx, row in df.iterrows():
        titulo = str(row["Nombre"])

        # 1. Corrección de Marca y Almacenamiento
        marca = extraer_marca_corregida(titulo)
        alm_gb, alm_tipo = extraer_almacenamiento_corregido(titulo)

        # 2. Extracción Determinista de Procesador
        cpu_m, nucleos, gen = identificar_procesador_completo(titulo)

        marcas_list.append(marca)
        alm_gb_list.append(alm_gb if alm_gb else row.get("Almacenamiento_GB"))
        alm_tipo_list.append(alm_tipo)
        modelos_cpu.append(cpu_m)
        nucleos_list.append(nucleos)
        gen_list.append(gen)

    # Asignar columnas corregidas y enriquecidas
    df["Marca"] = marcas_list
    df["Almacenamiento_GB"] = alm_gb_list
    df["Tipo_Almacenamiento"] = alm_tipo_list
    df["Procesador_Modelo"] = modelos_cpu
    df["Numero_Nucleos"] = nucleos_list
    df["Generacion_CPU"] = gen_list

    # Imputación final para Pantalla si faltara
    if "Pantalla_Pulgadas" in df.columns:
        df["Pantalla_Pulgadas"] = df["Pantalla_Pulgadas"].fillna(15.6)

    df.to_csv(csv_salida, index=False)
    print(f"¡Éxito! Base enriquecida guardada en '{csv_salida}'.")
    return df


if __name__ == "__main__":
    df_resultado = enriquecer_base_datos()

¡Éxito! Base enriquecida guardada en 'laptops_base_enriquecida.csv'.
